In [27]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_core.runnables import RunnableParallel , RunnablePassthrough , RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

In [28]:
load_dotenv()

model = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite-preview')

embedding_model = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-2-preview')

In [2]:
loader = PyPDFLoader('DEV RESUME.pdf')

In [3]:
doc = loader.load()

In [5]:
len(doc[0].page_content)

2509

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600 , 
    chunk_overlap = 100
)

In [8]:
chunk = splitter.split_documents(doc)

In [15]:
vector_store = FAISS.from_documents(
    embedding=embedding_model ,
    documents = chunk
)

In [34]:
retriever = vector_store.as_retriever(
    search_type = 'mmr' , 
    search_kwargs = {'k' :1}
)

In [19]:
def format(retrieved_documents):

    text_context = '\n\n'.join(d.page_content for d in retrieved_documents)

    return text_context

In [23]:
parallel_chain = RunnableParallel({
    'context' :  retriever | RunnableLambda(format) , 
    'query' : RunnablePassthrough()
})

In [26]:
prompt = PromptTemplate(
    template='Hey assistant, i am giving a context just take that context and answer the given query, if context is insufficient then just say i don\'t know \n context -- > {context} \n query ---- > {query}' ,
    input_variables=['context' , 'query']
)

In [29]:
parser = StrOutputParser()

In [ ]:
chain = parallel_chain | prompt | model | parser 

In [57]:
query = 'what are my projects'

In [58]:
result = chain.invoke(query)

In [59]:
print(result)

Based on the context provided, your projects are:

1.  **Medical Image Classification System (2025):** A deep learning project using CNNs, Python, TensorFlow, and Keras to classify medical images into disease categories.
2.  **Semantic Search Engine using Sentence Transformers (2025):** An NLP project using Python, Sentence Transformers, and Scikit-learn that implements cosine similarity for top-k document retrieval and uses pre-trained models to generate dense text embeddings.
